In [ ]:
%pip install langchain

In [ ]:
%pip install langchain-groq

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    #tempearature=0,
    groq_api_key=os.environ['GROQ_API_KEY'],  # never hardcode keys
    model_name="llama-3.3-70b-versatile")

response= llm.invoke("Who is Sushmoy Datta who is a student at IIT ISM Dhanbad and IIUC")
print(response.content)

In [ ]:
%pip install chromadb

In [ ]:
import chromadb 

client= chromadb.Client()
collection= client.create_collection(name="cc_collection")


In [ ]:
collection.add(
    documents=[
        'This document is about Dhaka',
        'This document is about Texas'
    ],
    ids=['id3','id4'],
    
    metadatas=[
        {"url": "https://en.wikipedia.org/wiki/Dhaka"},
        {"url": "https://en.wikipedia.org/wiki/Texas"} ]
)   

In [ ]:
all_docs= collection.get()
all_docs

In [ ]:
documents= collection.get(ids=['id2'])
documents

In [ ]:
results= collection.query(
    query_texts=['Query is about Dani Daniels'],
    n_results=2
)
results

In [ ]:
%pip install -qU langchain_community beautifulsoup4


In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/software-engineer/job/R-61123")
page_data = loader.load().pop()
print(page_data)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_extract= PromptTemplate.from_template(
    """
    #SCRAPED TEXT FROM WEBSITE:
    {page_data}
    ###INSTRUCTION:
    The scraped text is from the career's page of a website.
    Your job is to extract the job postings and return them in JSON fromat contining the following keys: 'role', 'experience', 'skills' and 'description'.
    Only return the valid JSON.
    ### VAID JSON (NO PREAMBLE):
    """
)
chain_extract= prompt_extract | llm
res= chain_extract.invoke(input={'page_data':page_data})
print(res.content)
type(res.content)

    

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser= JsonOutputParser()
json_res= json_parser.parse(res.content)
json_res

In [ ]:
type(json_res)

In [ ]:
import pandas as pd 
df= pd.read_csv("Sushmoy_Resume_Extracted.csv")
df

In [ ]:
import uuid
import chromadb

client= chromadb.PersistentClient('vectorstore')
collection= client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Field"],
                       metadatas={"Value": row["Value"]},
                       ids=[str(uuid.uuid4())])

In [ ]:
values = collection.query(query_texts=["Skills"],
                        n_results=5).get('metadatas', [])
values

In [ ]:
job

In [ ]:
job= json_res
job['skills']

In [ ]:
from langchain_core.prompts import PromptTemplate
Prompt_Coverletter = PromptTemplate.from_template(
    """
    ### JOB POSTING:
    {job_posting}

    ### CANDIDATE PROFILE:
    Name: Sushmoy Datta  
    Email: sushmoi007@gmail.com  
    Mobile: +91-8981022901  
    LinkedIn: linkedin.com/in/sushmoy-datta  

    Education:
    - M.Tech in Electronics & Communication Engineering, IIT (ISM) Dhanbad (2024–Present)
    - B.Tech in Electronics & Communication Engineering, IIUC Bangladesh (2018–2022)

    Skills:
    - Strong in Python, VHDL, Verilog, C/C++, SQL
    - Tools: Xilinx Vivado, Cadence Virtuoso, LTSpice, MATLAB, Google Colab
    - Domains: VLSI Design, Machine Learning, Signal Processing, Digital/Analog Circuit Design

    Experience:
    - Teaching Assistant at IIT ISM (Basic Electronics Lab)
    - Brand Executive at Teachnook (IIT ISM)
    - Business Development Intern at GAOTek Inc.

    Research & Projects:
    - M.Tech Thesis on IRS Element Selection using Online Machine Learning (FTRL, FTPL algorithms)
    - Projects include 2-stage Op-Amp design, SRAM cell, FSM on FPGA, MIPS32 Processor, and ML/NLP implementations in Python

   ### INSTRUCTION:
    Write a compelling and personalized cover letter based on the job posting and candidate profile. The letter should reflect Sushmoy’s technical proficiency, academic rigor, and motivation to contribute meaningfully to the employer’s mission.

    The cover letter must:
    - Begin with a strong opening that grabs attention and shows genuine interest in the role or company
    - Clearly connect the candidate’s background (Python, VHDL, AI/ML, VLSI) to the specific responsibilities or technologies mentioned in the job description
    - Highlight 1–2 standout academic or research projects that demonstrate initiative, problem-solving, or innovation
    - Reflect alignment with the company’s values, vision, or goals (if provided in the job post)
    - Use professional yet engaging language to convey enthusiasm, adaptability, and a growth mindset
    - End with a forward-looking statement requesting an opportunity for further discussion or interview

    ### COVER LETTER BODY (do not include salutation or closing signature; just the main content of the letter in professional tone):
    """
)
chain_Coverletter= Prompt_Coverletter | llm
res= chain_Coverletter.invoke({"job_posting": str(job)})
print(res.content)